# TIP/IEF 市场差：原生 Qlib 来源映射

回溯发行人 NAV 与除息字段只作有条件的市场代理，按冻结 Qlib 会话滞后两个交易日；不读取 SOXX 结果或拟合。

In [1]:
import hashlib,json,os,pathlib,shlex
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.soxx_issuer_fund_download import read_historical
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/stage1_tip_ief_market_spread_qlib_map_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
source=root/src['issuer_source_card'];summary=json.loads((source/'summary.json').read_text());assert sha(source/'summary.json')==src['issuer_summary_sha256'] and summary['NAS_source_bundle_sha256']==src['current_source_NAS_sha256'] and summary['vintage_2015_NAS_sha256']==src['2015_vintage_NAS_sha256']
assert sha(root/src['Qlib_daily_provider']/'qlib_store_manifest.json')==src['Qlib_daily_manifest_sha256']
raw=root/'data/snapshots/stage1_tip_ief_issuer_market_proxy_source_20261023';frames={}
for symbol in ('TIP','IEF'):
 path=raw/(symbol+'_fund_download.response');assert sha(path)==src[symbol+'_raw_sha256'];frame,_=read_historical(path);frames[symbol]=frame
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['Qlib_daily_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
calendar=pd.DatetimeIndex(D.calendar(start_time='2010-12-01',end_time='2026-10-07',freq='day')).normalize();assert calendar.is_unique and calendar.is_monotonic_increasing
returns={};missing_source={}
for symbol,frame in frames.items():
 nav=frame.nav_per_share.reindex(calendar).astype(float);raw_dist=frame.ex_dividends.replace('--','0');dist=pd.to_numeric(raw_dist,errors='raise').reindex(calendar).astype(float)
 valid=nav.gt(0)&nav.shift(1).gt(0)&dist.notna()&(nav+dist).gt(0)
 ret=pd.Series(np.where(valid,np.log((nav+dist)/nav.shift(1)),np.nan),index=calendar)
 returns[symbol]=ret;missing_source[symbol]=int((~valid).sum())
spread=returns['TIP']-returns['IEF'];values=[];underlying_source_end=[]
for i,day in enumerate(calendar):
 if i<6:values.append(np.nan);underlying_source_end.append(None);continue
 window=spread.iloc[i-6:i-1]
 values.append(float(window.sum()) if len(window)==5 and window.notna().all() else np.nan)
 underlying_source_end.append(str(calendar[i-2].date()))
source_map=pd.DataFrame({'signal_date':calendar,'tip_minus_ief_5session_log_return_tminus2':values,'last_source_session':underlying_source_end})
source_map['known']=source_map.tip_minus_ief_5session_log_return_tminus2.notna()
train=source_map.loc[source_map.signal_date.between('2011-01-03','2022-12-30')].copy();held=source_map.loc[source_map.signal_date.between('2023-01-03','2026-10-07')].copy()
assert len(train)>=3000 and len(held)==944
assert all(pd.Timestamp(end)<date for end,date in zip(source_map.last_source_session.dropna(),source_map.loc[source_map.last_source_session.notna(),'signal_date']))
parent=root/src['original39_card']
with R.uri_context(str(parent/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['original39_experiment_id'],recorder_id=src['original39_recorder_id']);assert rec.status=='FINISHED'
 model=rec.load_object('models/base_outer_71');features=rec.load_object('inputs/features')
names=list(model.base_features);assert len(names)==39 and set(names).issubset(features.columns)
known=train.loc[train.known].set_index('signal_date').tip_minus_ief_5session_log_return_tminus2
corr=features.loc[known.index,names].apply(lambda col:col.corr(known,method='spearman')).abs().sort_values(ascending=False)
assert len(corr)==39 and np.isfinite(known.to_numpy(float)).all()
static=pd.concat({'feature':source_map.set_index('signal_date')[['tip_minus_ief_5session_log_return_tminus2']],'meta':source_map.set_index('signal_date')[['known']]},axis=1)
static.index=pd.MultiIndex.from_arrays([static.index,['SOXL_SOXX_CASH']*len(static)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'training':('2011-01-03','2022-12-30'),'held':('2023-01-03','2026-10-07')})
assert len(dataset.prepare('training',col_set=['feature','meta']))==len(train) and len(dataset.prepare('held',col_set=['feature','meta']))==944
result={'decision':'conditional_retrospective_TIP_IEF_market_proxy_Qlib_map_source_only','plan_sha256':sha(card/'plan.json'),'Qlib_calendar_sessions':len(calendar),'train_calendar_sessions':len(train),'train_known':int(train.known.sum()),'train_unknown_dates':[str(x.date()) for x in train.loc[~train.known,'signal_date']],'held_calendar_sessions':len(held),'held_known':int(held.known.sum()),'held_unknown_dates':[str(x.date()) for x in held.loc[~held.known,'signal_date']],'TIP_one_session_source_missing':missing_source['TIP'],'IEF_one_session_source_missing':missing_source['IEF'],'max_abs_train_spearman_original39':float(corr.iloc[0]),'max_abs_train_spearman_feature':str(corr.index[0]),'exact_duplicate_original_input':bool(corr.iloc[0]>=0.95),'historical_first_public_qualified':False,'2016_2022_revision_qualified':False,'NAV_ex_dividend_semantics_fully_qualified':False,'TIP_IEF_spread_is_real_yield':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();dataset.config(dump_all=True,recursive=True)
 rec.save_objects(artifact_path='inputs',plan=plan,dataset=dataset)
 rec.save_objects(artifact_path='analysis',source_map=source_map,result=result,training_correlations=corr.to_dict())
 rec.log_metrics(train_known=result['train_known'],held_known=result['held_known'],max_abs_train_spearman=result['max_abs_train_spearman_original39'],model_fits=0,native_accounts=0)
 rid,eid=rec.id,str(rec.experiment_id)
result['recorder_id']=rid;result['experiment_id']=eid;result['recorder_uri']=str(card/'qlib_recorder');(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);settings=_settings(root);remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256'];result['NAS_Recorder_sha256']=remote;result['NAS_remote_hash_verified']=True;(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({k:v for k,v in result.items() if k not in ('train_unknown_dates','held_unknown_dates')},ensure_ascii=False,indent=2));print('unknown counts',len(result['train_unknown_dates']),len(result['held_unknown_dates']))


{
  "decision": "conditional_retrospective_TIP_IEF_market_proxy_Qlib_map_source_only",
  "plan_sha256": "59bffa82446b79801ecfa34e8391cbbfecb4286b36922a09faded99a071398bf",
  "Qlib_calendar_sessions": 3986,
  "train_calendar_sessions": 3020,
  "train_known": 3020,
  "held_calendar_sessions": 944,
  "held_known": 944,
  "TIP_one_session_source_missing": 1,
  "IEF_one_session_source_missing": 1,
  "max_abs_train_spearman_original39": 0.2218188565767565,
  "max_abs_train_spearman_feature": "soxx_ret10",
  "exact_duplicate_original_input": false,
  "historical_first_public_qualified": false,
  "2016_2022_revision_qualified": false,
  "NAV_ex_dividend_semantics_fully_qualified": false,
  "TIP_IEF_spread_is_real_yield": false,
  "SOXX_outcome_reads": 0,
  "model_fits": 0,
  "native_accounts": 0,
  "recorder_id": "88e1e6c4213a4fb59b5a90985213c453",
  "experiment_id": "283492540310725322",
  "recorder_uri": "/Users/rziiiii/dev/workspace/research/stage1_tip_ief_market_spread_qlib_map_20261023/ql

## 中文结论

Qlib 决策日前 t−6 至 t−2 的 TIP−IEF 五日来源坐标，训练3020/3020、复用评价944/944 已知；与原39项最大训练期绝对 Spearman 0.2218。原始双工作簿及2015版本已 NAS 备份，日级坐标原生 Recorder 亦 NAS 冷读一致。逐日首次公开、2016–2022修订和分红口径仍未知；零 SOXX 标签、拟合与账户。